# Sidecar to Ambient Upgrade

Migrate a running petstore app from Istio **sidecar** mode to **ambient**, one
namespace at a time, on the Solo distribution of Istio installed by the Gloo
Operator. One kind cluster. The app keeps serving the whole way, a load
generator proves it, and the last namespace rolls back with a single label.

What this walks through:

- Solo Istio in **sidecar** mode from one `ServiceMeshController` CR.
- A petstore app with an **L7** namespace (a `catalog` v1/v2 canary via
  DestinationRule + VirtualService, an HTTP AuthorizationPolicy, VS retries and
  timeout) and an **L4-only** namespace (a TCP data store, STRICT mTLS, an
  identity-based L4 AuthorizationPolicy).
- Turning the mesh ambient by editing **one field**.
- Migrating the L4 namespace with **no waypoint**, and the L7 namespace **behind
  a waypoint**, with the DestinationRule and VirtualService still working.
- The Solo capability that makes a mixed fleet safe: sidecar and ingress traffic
  routed **through the waypoint**.
- Moving the canary from DR/VS subsets to a Gateway-API **HTTPRoute**.
- A single-label **rollback**, and a check-by-check list at the end.

This is an Enterprise lab. The mixed-fleet piece (a sidecar or the ingress
gateway sending traffic through a waypoint) is a Solo-distribution behaviour,
`ENABLE_WAYPOINT_INTEROP`, on by default on the Solo images. Community Istio does
not do it, so a waypoint's L7 policy would be silently unenforced for sidecar
callers during a mixed-fleet migration.

## Before you start

Kernel: this notebook runs shell. In Cursor, pick the Bash kernel (Select
Kernel, top-right, Jupyter Kernel, Bash). If you do not have it:
`pip install bash_kernel && python -m bash_kernel.install`.

You need Docker, `kind`, `kubectl`, `helm` and `gcloud` (authenticated so the
Solo Istio images pull from `us-docker.pkg.dev/soloio-img/istio`), plus a
`SOLO_ISTIO_LICENSE_KEY`. Point `SECRETS_FILE` at a shell file that exports it,
or export it directly. Set that up in the next cell.

In [ ]:
# Point this at a file that exports SOLO_ISTIO_LICENSE_KEY (or export it yourself)
export SECRETS_FILE="${SECRETS_FILE:-$HOME/secrets/secrets-envs.sh}"

# Context + a short kubectl helper reused by later cells (the Bash kernel keeps
# one shell session, so these persist).
export CTX=kind-ambient-migration
kc() { kubectl --context "$CTX" "$@"; }
echo "SECRETS_FILE=$SECRETS_FILE"

## 1. Cluster and Solo Istio in sidecar mode

One kind cluster (a control-plane and two workers, so ztunnel later has real
nodes to spread across). The Gloo Operator installs Solo Istio from a single
`ServiceMeshController` CR with `dataplaneMode: Sidecar`. This is the lifecycle
model that retires the end-of-life upstream operator: one object, and upgrades
are a field edit.

`01-cluster.sh` also handles two kind-specific things the script comments
explain: it removes the Gateway-API `safe-upgrades` ValidatingAdmissionPolicy so
the operator can manage its own bundled CRDs, and it loads the Solo images with
`docker save --platform` so Docker's containerd image store does not trip kind.

In [ ]:
./scripts/01-cluster.sh

The mesh is up in sidecar mode. Note what ambient mode is **not** here: there is
an `istio-cni` DaemonSet (it does the traffic redirection) but **no ztunnel**
yet. istiod carries `ENABLE_WAYPOINT_INTEROP=true`, the Solo default that makes
the mixed-fleet step later work.

In [ ]:
kc -n istio-system get deploy,ds
echo "--- the Solo interop flag on istiod ---"
kc -n istio-system get deploy istiod-gloo \
  -o jsonpath='{range .spec.template.spec.containers[0].env[*]}{.name}={.value}{"\n"}{end}' | grep WAYPOINT_INTEROP

## 2. The petstore app in sidecar mode

Three namespaces, all starting with `istio-injection=enabled`:

- **petstore** (L7): a `catalog` Service fronting v1 and v2 Deployments. A
  DestinationRule declares the subsets, a VirtualService splits traffic and adds
  retries and a timeout, and an HTTP AuthorizationPolicy allows only GET/HEAD.
- **petstore-data** (L4 only): a Redis reached over plain TCP, with STRICT mTLS
  and an L4 AuthorizationPolicy that lets only the catalog identity connect.
- **petstore-legacy**: a `checkout` client and a `fortio` load generator we
  leave on sidecars throughout, so they double as mixed-fleet callers.

In [ ]:
./scripts/02-apps.sh

Baseline in sidecar mode: every pod is 2/2 (app + sidecar), the canary is 100% v1, GET is allowed and DELETE is denied, and the L4 rule lets the catalog identity into Redis but no one else.

In [ ]:
echo "=== pods: 2/2 = app + sidecar ==="
kc get pods -n petstore
echo "=== canary (in-mesh): 100% v1 ==="
kc -n petstore-legacy exec deploy/checkout -c checkout -- \
  sh -c 'for i in $(seq 1 12); do curl -s http://catalog.petstore/; echo; done' | grep -o 'v[12]' | sort | uniq -c
echo "=== L7 authz: GET 200, DELETE 403 ==="
kc -n petstore-legacy exec deploy/checkout -c checkout -- sh -c \
  'echo -n "GET   -> "; curl -s -o /dev/null -w "%{http_code}\n" http://catalog.petstore/;
   echo -n "DELETE-> "; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE http://catalog.petstore/'
echo "=== L4 allow (catalog identity) ==="
kc -n petstore exec deploy/data-client -c client -- sh -c 'timeout 5 redis-cli -h redis.petstore-data ping'

### The one decision per namespace: L4 or L7

This is the whole migration in one question. ztunnel alone (no waypoint) covers
everything at L4: STRICT mTLS, and AuthorizationPolicy by principal, namespace or
ipBlock. You only need a **waypoint** when a namespace has L7 concerns: an HTTP
AuthorizationPolicy on methods, paths or headers, JWT RequestAuthentication, or
VirtualService/HTTPRoute behaviour like retries, timeouts and routing.

So `petstore-data` (identity-based TCP rule only) needs **no waypoint**, and
`petstore` (HTTP method rule + VS routing) needs **one before it is enrolled**.
Enrol a namespace that has an L7 policy without a waypoint and ztunnel fails
safe: it denies all traffic to that workload.

## 3. Turn the mesh ambient (one field)

Re-apply the same `ServiceMeshController` with `dataplaneMode: Ambient`. The
operator adds the `ztunnel` DaemonSet and switches istiod into ambient mode.
Running sidecar workloads are not touched, so the flip is zero-downtime and
nothing is enrolled yet.

In [ ]:
./scripts/03-flip-ambient.sh

In [ ]:
echo "=== ztunnel is now present alongside istio-cni ==="
kc -n istio-system get ds
echo "=== sidecar apps untouched, and fortio sees no errors across the flip ==="
kc get pods -n petstore | grep catalog
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 800 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

## 4. Migrate the L4-only namespace (no waypoint)

`petstore-data` has only an identity-based TCP rule, so ztunnel does all of it.
Enrol it, stop injection, restart. Redis comes back 1/1 with no sidecar and the
same L4 rule is still enforced, now by ztunnel.

In [ ]:
./scripts/04-migrate-l4.sh

The rule still holds with no waypoint anywhere in the namespace: the catalog identity is allowed, a different identity is denied at L4 (`redis-cli` reports the connection closed, which is the RBAC reset).

In [ ]:
echo "=== waypoint in petstore-data? (none) ==="
kc -n petstore-data get gateway 2>/dev/null || echo "  (none)"
echo "=== allowed: data-client (catalog identity) ==="
kc -n petstore exec deploy/data-client -c client -- sh -c 'timeout 5 redis-cli -h redis.petstore-data ping'
echo "=== denied: a pod running as the checkout identity ==="
kc -n petstore-legacy delete pod l4test --ignore-not-found >/dev/null 2>&1
kc -n petstore-legacy run l4test --image=redis:7-alpine \
  --overrides='{"spec":{"serviceAccountName":"checkout"}}' --restart=Never --command -- sleep 60 >/dev/null 2>&1
kc -n petstore-legacy wait --for=condition=Ready pod/l4test --timeout=40s >/dev/null 2>&1
kc -n petstore-legacy exec l4test -c l4test -- sh -c 'timeout 6 redis-cli -h redis.petstore-data ping 2>&1'
kc -n petstore-legacy delete pod l4test --ignore-not-found >/dev/null 2>&1

## 5. Migrate the L7 namespace (behind a waypoint)

`petstore` has an HTTP policy and VS routing, so the order matters. The script
does it safely: deploy the waypoint first, translate the AuthorizationPolicy from
`selector` to `targetRefs` and bind catalog to the waypoint, then enrol the
namespace and drop the old selector policy. The DestinationRule and
VirtualService keep working, now enforced on the waypoint instead of a sidecar.

In [ ]:
./scripts/05-migrate-l7.sh

The interesting checks. The canary still routes at the waypoint, and it is a real
split, not a default: shift the VirtualService weight to 50/50 and both versions
appear.

Note on subsets: the Solo docs say ambient does not support DestinationRule
subsets even with a waypoint. Tested here on 1.29.3-solo, they **do** route at
the waypoint. So keeping the DR/VS subset canary is viable on this version;
`HTTPRoute` (step 7) is still the recommended forward direction, so confirm this
on the exact version you deploy.

In [ ]:
echo "=== canary at the waypoint: 100% v1 ==="
kc -n petstore exec deploy/data-client -c client -- \
  sh -c 'for i in $(seq 1 20); do wget -qO- http://catalog.petstore/ 2>/dev/null; echo; done' | grep -o 'v[12]' | sort | uniq -c
echo "=== shift VS to 50/50 — a real split proves subsets route at the waypoint ==="
kc -n petstore patch virtualservice catalog --type=json -p='[
  {"op":"replace","path":"/spec/http/0/route/0/weight","value":50},
  {"op":"replace","path":"/spec/http/0/route/1/weight","value":50}]' >/dev/null
sleep 3
kc -n petstore exec deploy/data-client -c client -- \
  sh -c 'for i in $(seq 1 40); do wget -qO- http://catalog.petstore/ 2>/dev/null; echo; done' | grep -o 'v[12]' | sort | uniq -c
kc -n petstore patch virtualservice catalog --type=json -p='[
  {"op":"replace","path":"/spec/http/0/route/0/weight","value":100},
  {"op":"replace","path":"/spec/http/0/route/1/weight","value":0}]' >/dev/null
echo "=== the still-sidecar checkout is enforced by the waypoint too (Solo interop) ==="
kc -n petstore-legacy exec deploy/checkout -c checkout -- sh -c \
  'echo -n "GET   -> "; curl -s -o /dev/null -w "%{http_code}\n" http://catalog.petstore/;
   echo -n "DELETE-> "; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE http://catalog.petstore/'
echo "=== fortio across the L7 cut ==="
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 800 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

## 6. The Solo capability: sidecar and ingress through the waypoint

The in-mesh sidecar checkout was already routed through the waypoint the moment
catalog got `istio.io/use-waypoint`. That is the Solo interop working for
sidecars automatically. The Istio **ingress gateway** needs one explicit label,
`istio.io/ingress-use-waypoint`, to do the same.

Before the label, the ingress bypasses the waypoint: a DELETE through the ingress
is allowed and the canary is ignored (traffic hits both versions). After it, the
waypoint's authz and the canary apply to north-south traffic too. This is the
part community Istio cannot do, and it is why a mixed sidecar/ambient fleet is
only safe on the Solo images.

In [ ]:
echo "=== BEFORE: ingress bypasses the waypoint ==="
echo -n "ingress DELETE -> "; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE -H 'Host: petstore.local' http://localhost:18080/
echo -n "ingress versions -> "; for i in $(seq 1 20); do curl -s -H 'Host: petstore.local' http://localhost:18080/; echo; done | grep -o 'v[12]' | sort | uniq -c | tr '\n' ' '; echo

./scripts/06-solo-interop.sh
sleep 5

echo "=== AFTER: ingress routed through the waypoint (authz + canary apply) ==="
echo -n "ingress DELETE -> "; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE -H 'Host: petstore.local' http://localhost:18080/
echo -n "ingress versions -> "; for i in $(seq 1 20); do curl -s -H 'Host: petstore.local' http://localhost:18080/; echo; done | grep -o 'v[12]' | sort | uniq -c | tr '\n' ' '; echo

## 7. Modernize the canary: DR/VS subsets to HTTPRoute

Subsets work at the waypoint on this version, but the forward direction is the
Gateway API. The pattern: give each version a real Service, and split with an
`HTTPRoute` whose `parentRef` is the original catalog Service, so callers keep
using the same hostname. Shift traffic by editing `backendRef` weights, which is
what Argo Rollouts drives. Keep the DestinationRule for its traffic policy, drop
its subsets, and retire the VirtualService.

In [ ]:
./scripts/07-httproute.sh

In [ ]:
echo "=== canary now on the HTTPRoute — shift to 30/70 ==="
kc -n petstore patch httproute catalog --type=json -p='[
  {"op":"replace","path":"/spec/rules/0/backendRefs/0/weight","value":30},
  {"op":"replace","path":"/spec/rules/0/backendRefs/1/weight","value":70}]' >/dev/null
sleep 3
kc -n petstore exec deploy/data-client -c client -- \
  sh -c 'for i in $(seq 1 40); do wget -qO- http://catalog.petstore/ 2>/dev/null; echo; done' | grep -o 'v[12]' | sort | uniq -c
kc -n petstore patch httproute catalog --type=json -p='[
  {"op":"replace","path":"/spec/rules/0/backendRefs/0/weight","value":100},
  {"op":"replace","path":"/spec/rules/0/backendRefs/1/weight","value":0}]' >/dev/null
echo "=== fortio across the HTTPRoute cutover ==="
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 800 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

## 8. Rollback

The safety net. Any namespace goes back to sidecars with a single label flip:
turn ambient off, injection on, restart. The workloads keep running and the L4
rule stays enforced across the round trip. Shown on `petstore-data`.

In [ ]:
./scripts/rollback.sh petstore-data
echo "=== redis back to 2/2 (sidecar) and the L4 rule still holds ==="
kc get pods -n petstore-data
kc -n petstore exec deploy/data-client -c client -- sh -c 'timeout 5 redis-cli -h redis.petstore-data ping'

## Check-by-check migration checklist

The order that keeps it zero-downtime and reversible. Per cluster, once:

- [ ] Solo images everywhere in the mesh (the mixed-fleet interop needs them).
- [ ] Gloo Operator installed; mesh driven by a `ServiceMeshController`.
- [ ] Gateway API CRDs present (waypoints and HTTPRoute need them).
- [ ] Turn the mesh ambient: `dataplaneMode: Ambient`. Confirm `ztunnel` and
      `istio-cni` are Ready and sidecar workloads are untouched.

Per namespace, in this order:

- [ ] **Audit L4 vs L7.** List the namespace's policies. Any HTTP authz
      (methods/paths/headers), JWT RequestAuthentication, or VS/HTTPRoute L7
      behaviour means it needs a waypoint. Identity-only rules (principal,
      namespace, ipBlock) do not.
- [ ] **Audit for `PeerAuthentication mode: DISABLE`.** It has no ambient
      equivalent (HBONE is always mTLS). Keep such a workload out of the mesh, or
      move the plaintext consumer off the in-mesh path, before enrolling.
- [ ] **If L7: deploy the waypoint first** (`gatewayClassName: istio-waypoint`),
      wait for Programmed.
- [ ] **Translate policies** `selector` to `targetRefs` (Service or Gateway).
      Apply the new ones before removing the old.
- [ ] **Bind the workload** to the waypoint (`istio.io/use-waypoint`).
- [ ] **Enrol**: `istio.io/dataplane-mode=ambient`, remove `istio-injection`.
- [ ] **Remove the old selector L7 policy** (it trips ztunnel's fail-safe deny
      in ambient), then rolling-restart so pods come back with no sidecar.
- [ ] **For north-south**: set `istio.io/ingress-use-waypoint=true` on the
      Service so the ingress gateway routes through the waypoint. Do this
      **before** any from-waypoint lockdown, or ingress traffic breaks.
- [ ] **Verify**: canary/routing, L7 authz, L4 allow/deny, and a load run with no
      errors. Watch it from a sidecar caller too (the mixed-fleet path).
- [ ] **Roll back** if anything is off: `dataplane-mode-`, re-add
      `istio-injection`, restart. No flag day.
- [ ] **Clean up** the old sidecar-era policies once traffic is confirmed on the
      waypoint. Keep any L4 from-waypoint rules.

Optional, once the fleet is ambient:

- [ ] Modernize subset canaries to per-version Services + `HTTPRoute` (Argo
      Rollouts drives the weights).
- [ ] Retire `Sidecar` egress scoping and `exportTo` (no ambient equivalent;
      ztunnel makes all services visible).

## Clean up

In [ ]:
./demo-scripts/reset.sh